# **Neptune — Experiment Tracking**

---

## Why an Experiment Tracker

When you run dozens of training jobs, you need one place that records *for every run*: the hyperparameters, the metrics over time, the resulting artifacts (checkpoints, plots), and the environment (git SHA, dependencies). **Neptune** is a hosted (or self-hosted) metadata store + dashboard built for exactly this. Each training job becomes a **run** you can later filter, sort, and compare side by side.

## Core Concepts

* **Project** — a workspace that groups related runs (e.g. `org/image-classifier`).
* **Run** — a single execution; you log everything into a namespaced, dict-like handle.
* **Fields / namespaces** — hierarchical keys like `train/loss`, `params/lr`, `model/checkpoint`.
* **Series vs. single values** — call `.append(value)` repeatedly to build a metric curve; assign once for a scalar/param.
* **Artifacts / files** — upload checkpoints, configs, images, or reference large files in object storage.

## Minimal Client Code

```python
import neptune

run = neptune.init_run(
    project="my-team/image-classifier",
    api_token="...",            # better: set NEPTUNE_API_TOKEN env var
    tags=["resnet50", "baseline"],
)

# log hyperparameters (a dict assigned once)
run["params"] = {"lr": 3e-4, "batch_size": 128, "epochs": 50}

for epoch in range(epochs):
    train_loss = train_one_epoch(...)
    val_acc = evaluate(...)
    # metric series — append builds the curve over time
    run["train/loss"].append(train_loss)
    run["val/accuracy"].append(val_acc)

# upload artifacts
run["model/checkpoint"].upload("best.pt")
run["diagnostics/confusion_matrix"].upload("cm.png")

run.stop()                      # flush & close (do this in a finally block)
```

Comparing runs is done in the web UI (or via the query API): pick a set of runs and Neptune overlays their `val/accuracy` curves and diffs their `params`.

## Neptune vs. MLflow vs. Weights & Biases

| | **Neptune** | **MLflow** | **W&B** |
|---|---|---|---|
| Hosting | SaaS or self-hosted | open-source, self-hosted (or managed) | SaaS (self-host on enterprise) |
| Strength | clean metadata store, fast run comparison, lightweight client | open standard, model registry, no vendor lock-in | rich dashboards, sweeps, collaboration |
| Logging API | `run["ns/key"].append(...)` | `mlflow.log_metric/param/artifact` | `wandb.log({...})` |
| Best when | you want managed tracking with strong UI and minimal setup | you want full ownership and an open model registry | you want polished viz + hyperparameter sweeps |

All three log the same fundamentals (params, metric series, artifacts); the choice is mostly about hosting model, UI, and ecosystem features rather than capability.

## References

* Neptune docs: https://docs.neptune.ai
* `init_run` / logging: https://docs.neptune.ai/logging/overview/
* PyTorch integration: https://docs.neptune.ai/integrations/pytorch/
* Comparison of trackers: https://neptune.ai/vs